# Diarka QAOA Portfolio — Notebook 06: Robustness and Risk-Aversion Sensitivity

**Week 3, Session 3.** Two stories in one notebook:

1. **Robustness (the rigour story).** The Session 3 simulator result and the Session 2 hardware result are both $n = 1$ — a single QAOA optimisation each. That's enough to demonstrate the pipeline but not enough to make any *statistical* claim. Here we re-run QAOA at $p = 1, 2, 3$ from **many random starting points** and report distributions, error bars, and worst-case performance.

2. **Risk-aversion sensitivity (the finance story).** The optimal portfolio depends on $q$ — the trade-off between expected return and variance. Here we sweep $q$ across a realistic range, watch how the classical optimum *shifts* between asset selections, and verify that QAOA tracks that shift correctly.

Both stories use existing infrastructure (`src/qaoa.py`, `src/encoding.py`); the new module is `src/experiments.py`, which wraps the orchestration and aggregation.

Expected runtime: roughly 8–12 minutes total on a modern laptop. The bulk is the simulator QAOA optimisations.

## 0. Setup

In [ ]:
from __future__ import annotations
from pathlib import Path
import sys
import warnings
import time

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import numpy as np
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore", category=Warning, module="scipy")

from qiskit.quantum_info import SparsePauliOp
from src.data import PortfolioStats
from src.experiments import run_multiseed_qaoa, run_risk_aversion_sweep

DATA_PROCESSED = ROOT / "data" / "processed"

# Cost Hamiltonian (Session 2) and ground-state metadata.
ham_data = np.load(DATA_PROCESSED / "hamiltonian.npz", allow_pickle=False)
H = SparsePauliOp.from_list(list(zip(ham_data["pauli_labels"], ham_data["pauli_coeffs"])))
ISING_OFFSET     = float(ham_data["ising_offset"])
E_GROUND         = float(ham_data["ground_energy"])
GROUND_BITS      = str(ham_data["ground_bitstring"])
GROUND_SELECTION = ham_data["ground_selection"].astype(int)
ENERGIES         = ham_data["sorted_energies"]
BITSTRINGS       = ham_data["sorted_bitstrings"]
E_MAX            = float(ENERGIES[-1])
TICKERS          = tuple(str(t) for t in ham_data["tickers"])

# Portfolio statistics (Session 1) — needed for the q-sweep, which rebuilds
# the QUBO from scratch at each new risk-aversion value.
stats = PortfolioStats.from_npz(DATA_PROCESSED / "portfolio_stats.npz")
BUDGET = 4

# Optional: hardware single-shot result (Session 2) for overlaying as a
# reference line on the robustness plots.
HW_PATH = DATA_PROCESSED / "hardware_results.npz"
if HW_PATH.exists():
    hw = np.load(HW_PATH, allow_pickle=False)
    HW_RATIO = float(hw["hw_ratio"])
    HW_BACKEND = str(hw["backend_name"])
    print(f"Hardware reference loaded: {HW_BACKEND}, approximation ratio = {HW_RATIO:.3f}")
else:
    HW_RATIO = None
    HW_BACKEND = None
    print("No hardware results found — will skip hardware reference line.")

print(f"Tickers:           {TICKERS}")
print(f"Ground-state E:    {E_GROUND:+.4f}")
print(f"Ground bitstring:  {GROUND_BITS}  ({', '.join(t for t, b in zip(TICKERS, GROUND_SELECTION) if b)})")

plt.rcParams.update({
    "figure.figsize": (10, 6),
    "axes.spines.top": False,
    "axes.spines.right": False,
})

## 1. Multi-seed robustness at $p = 1, 2, 3$

For each depth, run QAOA from a fresh random $(\gamma, \beta)$ initialisation, optimise with COBYLA, sample, and record the approximation ratio. Repeating this gives the distribution of what QAOA "typically achieves" at each depth — much more informative than the single-trajectory results from Session 3.

To keep runtime sane we use more seeds at the cheap depth (p=1) and fewer at the expensive ones:

| depth | seeds | maxiter | est. time |
|------|-------|---------|-----------|
| 1    | 10    | 150     | ~2 min    |
| 2    | 6     | 200     | ~3 min    |
| 3    | 6     | 300     | ~5 min    |

The output is a `MultiseedResult` dataclass per depth, holding the per-seed approximation ratio, ground-state probability, best-sampled energy, and iteration count.

In [ ]:
SCHEDULE = {
    1: dict(n_seeds=10, maxiter=150),
    2: dict(n_seeds=6,  maxiter=200),
    3: dict(n_seeds=6,  maxiter=300),
}
SHOTS = 2048

multiseed = {}

for p, cfg in SCHEDULE.items():
    t0 = time.time()
    print(f"Running p={p} with n_seeds={cfg['n_seeds']}, maxiter={cfg['maxiter']}...")
    ms = run_multiseed_qaoa(
        H, p,
        n_seeds=cfg["n_seeds"],
        ground_energy=E_GROUND,
        max_energy=E_MAX,
        ground_bitstring=GROUND_BITS,
        sorted_energies=ENERGIES,
        sorted_bitstrings=BITSTRINGS,
        shots=SHOTS,
        seed_offset=p,
        maxiter=cfg["maxiter"],
    )
    multiseed[p] = ms
    print(f"  done in {time.time() - t0:.1f}s")

In [ ]:
# Summary table.
print(f"{'p':>3}  {'seeds':>5}  {'ratio mean':>11}  {'ratio std':>10}  {'ratio min':>10}  {'ratio max':>10}  {'P(GS) max':>10}  {'best E':>9}")
print("-" * 85)
for p, ms in multiseed.items():
    s = ms.summary()
    print(f"{p:>3}  {s['n_seeds']:>5}  {s['ratio_mean']:>11.4f}  {s['ratio_std']:>10.4f}  "
          f"{s['ratio_min']:>10.4f}  {s['ratio_max']:>10.4f}  "
          f"{s['gs_prob_max']:>10.2%}  {s['best_sampled_min']:>+9.4f}")

### Robustness plot — distribution of approximation ratios

The headline figure for the rigour story. Each box covers the inter-quartile range across seeds at that depth; whiskers extend to min/max. The hardware single-shot ratio from Session 2 is overlaid as a horizontal line — it should sit somewhere within or near the simulator's seed distribution at $p = 1$, which is what "noise reduces but doesn't destroy the signal" looks like quantitatively.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

ps = sorted(multiseed.keys())
ratios_by_p = [multiseed[p].ratios for p in ps]
gs_probs_by_p = [multiseed[p].ground_probs for p in ps]

# Left panel: approximation-ratio box plot.
ax = axes[0]
bp = ax.boxplot(ratios_by_p, positions=ps, widths=0.45, patch_artist=True,
                showfliers=True, medianprops={"color": "white", "linewidth": 2})
for patch, c in zip(bp["boxes"], ["#1f77b4", "#ff7f0e", "#2ca02c"]):
    patch.set_facecolor(c)
    patch.set_alpha(0.7)

# Overlay per-seed points (jittered).
for p, ratios in zip(ps, ratios_by_p):
    jitter = np.random.default_rng(0).uniform(-0.12, 0.12, size=len(ratios))
    ax.scatter(np.full_like(ratios, p) + jitter, ratios, s=24,
               color="black", alpha=0.5, zorder=5)

if HW_RATIO is not None:
    ax.axhline(HW_RATIO, color="#d62728", linestyle="--", alpha=0.7,
               label=f"hardware ({HW_BACKEND}, p=1): r = {HW_RATIO:.3f}")
    ax.legend(loc="lower right")

ax.set_xlabel("QAOA depth p")
ax.set_ylabel("Approximation ratio  r")
ax.set_xticks(ps)
ax.set_xticklabels([f"p = {p}" for p in ps])
ax.set_title(f"Approximation ratio across random initialisations (N seeds varies by p)")
ax.grid(alpha=0.3)

# Right panel: ground-state probability box plot.
ax = axes[1]
bp = ax.boxplot(gs_probs_by_p, positions=ps, widths=0.45, patch_artist=True,
                showfliers=True, medianprops={"color": "white", "linewidth": 2})
for patch, c in zip(bp["boxes"], ["#1f77b4", "#ff7f0e", "#2ca02c"]):
    patch.set_facecolor(c)
    patch.set_alpha(0.7)
for p, probs in zip(ps, gs_probs_by_p):
    jitter = np.random.default_rng(1).uniform(-0.12, 0.12, size=len(probs))
    ax.scatter(np.full_like(probs, p) + jitter, probs, s=24,
               color="black", alpha=0.5, zorder=5)
ax.set_xlabel("QAOA depth p")
ax.set_ylabel("P(ground state) in 2048 shots")
ax.set_xticks(ps)
ax.set_xticklabels([f"p = {p}" for p in ps])
ax.set_title("Ground-state sampling probability")
ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()

## 2. Risk-aversion sensitivity at $p = 1$

The risk-aversion parameter $q$ controls how much the optimiser penalises variance relative to return:

$$\text{minimise} \quad q \cdot x^\top \Sigma x \;-\; \mu^\top x$$

- $q = 0$: pure return-maximising — pick the four assets with the highest expected return, ignore correlations.
- $q \to \infty$: pure variance-minimising — pick the four assets that together produce the lowest portfolio volatility, ignore returns.
- $q \approx 0.5$ (what we've used so far): a balanced trade-off, similar to a moderate-risk investor's mean-variance optimisation.

Sweeping $q$ produces a sequence of *different* QUBOs, hence different Ising Hamiltonians, hence different optimal portfolios. The question for QAOA: does it correctly identify the optimum across all of these problems, or does it break down in some regime?

In [ ]:
QS = [0.0, 0.1, 0.25, 0.5, 1.0, 2.0, 5.0]

t0 = time.time()
print(f"Sweeping q ∈ {QS} at p=1 with 3 seeds each...")
sweep = run_risk_aversion_sweep(
    stats.mu, stats.sigma,
    qs=QS, budget=BUDGET, p=1,
    n_seeds=3, shots=SHOTS, maxiter=150,
)
print(f"Risk-aversion sweep done in {time.time() - t0:.1f}s")
print()
print(f"{'q':>6}  {'ratio':>6}  {'return':>8}  {'risk':>8}  classical pick    qaoa pick      match")
print("-" * 80)
for r in sweep.rows:
    cl = "".join(str(b) for b in r.classical_selection)
    qa = "".join(str(b) for b in r.qaoa_selection)
    match = "✓" if np.array_equal(r.classical_selection, r.qaoa_selection) else "✗"
    print(f"{r.q:>6.2f}  {r.qaoa_ratio:>6.3f}  {r.classical_return:>+8.4f}  {r.classical_risk:>8.4f}  "
          f"{cl:>15}  {qa:>10}     {match}")

### Portfolio composition heatmap

Rows are risk-aversion values $q$ (low risk-aversion at top, high at bottom), columns are the eight FTSE tickers, colour shows whether that asset is included in the classical optimum.

This is the financial story compressed into one image: as $q$ increases, the optimiser moves from "growth basket" to "stability basket". Specific assets cross thresholds — some are *always* picked, some are *never* picked, some appear/disappear at specific risk-aversion levels. Those are the interesting transitions.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Left panel: classical selections as a heatmap.
ax = axes[0]
classical = sweep.classical_selections.astype(float)
im = ax.imshow(classical, cmap="Blues", aspect="auto", vmin=0, vmax=1)
ax.set_xticks(range(len(TICKERS)))
ax.set_xticklabels(TICKERS, rotation=45, ha="right")
ax.set_yticks(range(len(sweep.rows)))
ax.set_yticklabels([f"q = {r.q:.2f}" for r in sweep.rows])
ax.set_title("Classical-optimal portfolio composition vs risk aversion")

# Annotate cells.
for i in range(classical.shape[0]):
    for j in range(classical.shape[1]):
        ax.text(j, i, "●" if classical[i, j] > 0.5 else "",
                ha="center", va="center", color="white", fontsize=14)

# Right panel: q vs return / risk (the efficient frontier slice).
ax = axes[1]
ax.plot(sweep.classical_risks, sweep.classical_returns, "o-",
        color="#1f77b4", lw=2, markersize=8)
for r in sweep.rows:
    ax.annotate(f"q = {r.q:.2g}", (r.classical_risk, r.classical_return),
                xytext=(7, 5), textcoords="offset points", fontsize=9)
ax.set_xlabel("Portfolio risk  √(xᵀΣx)  (annualised)")
ax.set_ylabel("Portfolio return  μᵀx  (annualised)")
ax.set_title("Return vs risk along the classical-optimal frontier")
ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()

### QAOA accuracy across the risk axis

How well does QAOA track the classical optimum as we vary $q$? Two views:

1. **Approximation ratio** at each $q$ — a single number per problem instance.
2. **Selection agreement** — does QAOA's most-likely bitstring match the classical optimum exactly? A simple ✓/✗ verdict per row.

Both should be invariant to $q$ if QAOA is well-behaved, which is the claim we want to put in the writeup.

In [ ]:
fig, ax = plt.subplots(figsize=(11, 5))

ratios = sweep.qaoa_ratios
qs = sweep.qs

ax.plot(qs, ratios, "o-", color="#1f77b4", lw=2, markersize=10,
        label="QAOA (p=1, best of 3 seeds)")
ax.axhline(1.0, color="black", linestyle=":", alpha=0.4,
           label="r = 1 (exact ground state)")

# Mark agreement / disagreement with the classical optimum.
for r in sweep.rows:
    agreement = np.array_equal(r.classical_selection, r.qaoa_selection)
    marker = "*" if agreement else "x"
    color  = "#2ca02c" if agreement else "#d62728"
    ax.scatter([r.q], [r.qaoa_ratio + 0.012], marker=marker, s=180,
               color=color, zorder=10)

ax.set_xscale("symlog", linthresh=0.1)
ax.set_xlabel("Risk-aversion  q  (symmetric log)")
ax.set_ylabel("Approximation ratio")
ax.set_title("QAOA approximation ratio across the risk-aversion sweep\n"
             "(★ = QAOA picks the exact classical optimum;  ✗ = different selection)")
ax.legend(loc="lower right")
ax.grid(alpha=0.3)
ax.set_ylim(min(0.85, ratios.min() - 0.02), 1.05)
plt.tight_layout()
plt.show()

n_match = sum(np.array_equal(r.classical_selection, r.qaoa_selection) for r in sweep.rows)
print(f"QAOA agreement with classical optimum: {n_match} / {len(sweep.rows)} ({n_match / len(sweep.rows):.0%})")

## 3. Persist outputs

In [ ]:
out = {
    "ms_p1_ratios":       multiseed[1].ratios,
    "ms_p1_gs_probs":     multiseed[1].ground_probs,
    "ms_p1_best_sampled": multiseed[1].best_sampled,
    "ms_p2_ratios":       multiseed[2].ratios,
    "ms_p2_gs_probs":     multiseed[2].ground_probs,
    "ms_p2_best_sampled": multiseed[2].best_sampled,
    "ms_p3_ratios":       multiseed[3].ratios,
    "ms_p3_gs_probs":     multiseed[3].ground_probs,
    "ms_p3_best_sampled": multiseed[3].best_sampled,
    "sweep_qs":           sweep.qs,
    "sweep_classical_selections": sweep.classical_selections,
    "sweep_qaoa_selections":      sweep.qaoa_selections,
    "sweep_classical_returns":    sweep.classical_returns,
    "sweep_classical_risks":      sweep.classical_risks,
    "sweep_qaoa_ratios":          sweep.qaoa_ratios,
}
np.savez(DATA_PROCESSED / "robustness_and_sweep.npz", **out)
print(f"Saved {(DATA_PROCESSED / 'robustness_and_sweep.npz').relative_to(ROOT)}")

## Session 3 (Week 3) — wrap-up

Two defensible claims now have data behind them:

- **Rigour:** "Across N random initialisations at depth p, QAOA achieves an approximation ratio of $r = \bar r \pm \sigma_r$, with the minimum across seeds being $r_{\min}$" — at each p. These error bars convert the previous n=1 numbers into something quotable.

- **Domain:** "Sweeping the risk-aversion parameter across two orders of magnitude, QAOA correctly identifies the classical optimum in M of N problem instances" — with a heatmap of selected portfolios showing the actual transitions.

Both are now backed by single artefacts (`data/processed/robustness_and_sweep.npz`) and one or two LinkedIn-shareable plots each.

**Where Week 4 could go from here:**

- *Noise modelling.* Build a noisy simulator matching `ibm_marrakesh`'s reported gate fidelities and compare the noisy-simulator distribution to the hardware distribution from Session 2. If the noise model reproduces the hardware result, we have a predictive tool for future submissions.
- *Hardware at $p = 2$ or $p = 3$.* Submit deeper-circuit jobs and see whether the deeper-is-better simulator trend survives the additional CZ-gate noise.
- *Asset universe scaling.* Re-run the whole pipeline on a 12- or 16-asset problem to see where the noiseless approximation ratio starts to degrade.